Statistical analysis of raw data distribution, analysis of missing data, and detection of distribution drift between "Train/Test" using Adversarial Validation.

# **STAGE 1**

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add scripts directory to path for imports
sys.path.append(os.path.abspath("../scripts"))

from config import TRAIN_CSV, TEST_CSV, PATIENTS_CSV, CLINICS_CSV, VAL_SPLIT_DATE, STAGE1_OUT
import data_loader
from utils import validation

# Scientific plot settings
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
# Load all raw CSV files
df_train_raw = data_loader.load_raw_data(TRAIN_CSV)
df_test_raw = data_loader.load_raw_data(TEST_CSV)
df_patients = data_loader.load_raw_data(PATIENTS_CSV)
df_clinics = data_loader.load_raw_data(CLINICS_CSV)

# Perform static joins
df_train = data_loader.merge_with_lookups(df_train_raw, df_patients, df_clinics)
df_test = data_loader.merge_with_lookups(df_test_raw, df_patients, df_clinics)

print(f"Initial Train Shape: {df_train.shape}")
print(f"Initial Test Shape: {df_test.shape}")

In [ ]:
# Cleaning and Downcasting
for df in [df_train, df_test]:
    df = data_loader.convert_to_datetime(df, ['appointment_datetime', 'booking_datetime'])
    # Fix time leakage (Lead_Time check)
    df = data_loader.handle_lead_time_leakage(df)
    # Apply SMS logic and flags
    df = data_loader.process_sms_logic(df)
    df = data_loader.downcast_memory(df)

print(f"Min Lead Time in Train: {df_train['lead_time_hours'].min()}")

# **STAGE 2**

In [ ]:
# Sub-Train: Jan-Sep 2025 | Sub-Val: Oct 2025
sub_train, sub_val = validation.split_by_time(df_train, VAL_SPLIT_DATE)

print(f"Sub-Train Range: {sub_train['appointment_datetime'].min()} to {sub_train['appointment_datetime'].max()}")
print(f"Sub-Val Range: {sub_val['appointment_datetime'].min()} to {sub_val['appointment_datetime'].max()}")

In [ ]:
# Define numerical features to check for distribution shift
check_features = ['age', 'ses_score', 'lead_time_hours', 'sms_lead_hours']

# Prepare adversarial data
adv_data = validation.prepare_adversarial_data(sub_train, df_test, check_features)

# Run Validation
auc_score, feat_imp = validation.run_adversarial_validation(adv_data, check_features)

print(f"Adversarial AUC: {auc_score:.4f}")

# Visualize feature differences if AUC > 0.70
if auc_score > 0.70:
    print("CRITICAL: Significant distribution shift detected!")
    plt.figure(figsize=(10, 5))
    feat_imp.plot(kind='barh', color='salmon')
    plt.title("Features causing Train/Test deviation")
    plt.show()

In [ ]:
# Save to processed_data/stage1
sub_train.to_parquet(STAGE1_OUT / "train_base.parquet", index=False)
sub_val.to_parquet(STAGE1_OUT / "val_base.parquet", index=False)
df_test.to_parquet(STAGE1_OUT / "test_base.parquet", index=False)

print("Stage 1 & 2 base files saved as Parquet.")